In [ ]:
import os

# 1. 强制清理旧残留
print("正在清理旧文件...")
!rm -rf Diffusion-Illusions
!rm -rf master.zip

# 2. 克隆仓库
print("正在克隆仓库...")
!git clone https://github.com/RyannDaGreat/Diffusion-Illusions

# 3. 检查并安装依赖
if os.path.exists('Diffusion-Illusions'):
    print("✅ 仓库克隆成功！")
    %cd Diffusion-Illusions
    print("正在安装依赖 (红色警告请忽略)...")
    !pip install -r requirements.txt
    !pip install mediapy easydict "numpy<2.0"
    
    print("\n✅✅ 环境初始化全部完成！")
    print("⚠️⚠️ 现在的关键步骤：请点击上方菜单 'Runtime' -> 'Restart session' 重启运行时！")
else:
    print("❌❌ 克隆失败，请检查网络。")

In [ ]:
import os

repo_name = "Diffusion-Illusions"

# 检查当前是否已经在文件夹里了
if os.getcwd().endswith(repo_name):
    print(f"✅ 当前位置正确: {os.getcwd()}")
else:
    # 如果不在，就尝试进去
    if os.path.exists(repo_name):
        %cd {repo_name}
        print(f"✅ 已切换工作目录到: {os.getcwd()}")
    else:
        # 如果文件夹都不存在，说明之前的克隆没成功，重新克隆一下
        print("⚠️ 文件夹不存在，正在重新克隆...")
        !git clone https://github.com/RyannDaGreat/Diffusion-Illusions
        %cd {repo_name}
        print(f"✅ 克隆并切换完成: {os.getcwd()}")

In [ ]:
from rp import *
import numpy as np
import torch
import torch.nn as nn
import source.stable_diffusion as sd
from source.learnable_textures import LearnableImageFourier
from source.stable_diffusion_labels import NegativeLabel
from itertools import chain
import torchvision.transforms.functional as TF
from google.colab import files
from PIL import Image, ImageOps

# === 核心几何变换：折纸模拟 ===

def get_fold_indices(width):
    """计算折叠区域的分界线"""
    # 我们把图片分成三份：[左边 A] [中间 B] [右边 C]
    # 折叠逻辑：把 A 折过来盖住 B，边缘与 C 对接。
    # 最终看到的画面是：[A] + [C] (中间的 B 被藏在折痕里了)
    split1 = width // 3
    split2 = (width * 2) // 3
    return split1, split2

def simulate_fold(image_tensor):
    """
    模拟折纸效果
    输入: [1, 3, H, W] 的完整图片
    输出: [1, 3, H, W'] 折叠后的图片 (宽度变窄)
    """
    _, _, h, w = image_tensor.shape
    s1, s2 = get_fold_indices(w)
    
    # 提取左边部分 (Part A)
    part_left = image_tensor[:, :, :, :s1]
    # 提取右边部分 (Part C)
    part_right = image_tensor[:, :, :, s2:]
    
    # 拼接 (模拟折叠后 A 和 C 挨在了一起)
    folded = torch.cat([part_left, part_right], dim=3)
    return folded

def visualize_fold_lines(image_np):
    """辅助函数：在图上画虚线，显示折痕位置"""
    h, w, c = image_np.shape
    s1, s2 = get_fold_indices(w)
    
    img_copy = image_np.copy()
    # 画两条红线表示折痕，方便你看哪里需要折
    if c >= 3:
        img_copy[:, s1-2:s1+2, 0] = 1.0 # R
        img_copy[:, s1-2:s1+2, 1] = 0.0 # G
        img_copy[:, s1-2:s1+2, 2] = 0.0 # B
        
        img_copy[:, s2-2:s2+2, 0] = 1.0
        img_copy[:, s2-2:s2+2, 1] = 0.0
        img_copy[:, s2-2:s2+2, 2] = 0.0
    return img_copy

In [ ]:
# 初始化 GPU 和 模型
if 'model_sd' not in dir():
    print("正在加载 Stable Diffusion...")
    model_name = "CompVis/stable-diffusion-v1-4"
    gpu = rp.select_torch_device()
    model_sd = sd.StableDiffusion(gpu, model_name)
    device = model_sd.device
    print("模型加载完毕！")
else:
    print("模型已存在，跳过加载。")

In [ ]:
# 设定画布尺寸 (标准 SD 尺寸)
CANVAS_WIDTH = 512
CANVAS_HEIGHT = 512

s1, s2 = get_fold_indices(CANVAS_WIDTH)
FOLDED_WIDTH = s1 + (CANVAS_WIDTH - s2) # 折叠后的宽度

print(f"画布尺寸: {CANVAS_WIDTH}x{CANVAS_HEIGHT}")
print(f"折叠逻辑: 左侧 {s1}px + 右侧 {CANVAS_WIDTH-s2}px 将拼接在一起。中间 {s2-s1}px 被隐藏。")
print(f"折叠后预期宽度: {FOLDED_WIDTH}px")

print("\n>>> 请点击下方按钮上传你的'隐藏信息'图片 (例如文字 LOVE 或 某种图案) <<<\")
print("注意：程序会自动把你的图挤压成折叠后的比例，不需要手动调整。")
uploaded = files.upload()

if uploaded:
    filename = next(iter(uploaded))
    
    # 1. 读取并缩放到折叠后的尺寸 (关键步骤)
    target_pil = Image.open(filename).convert('RGB')
    # 强制缩放到 FOLDED_WIDTH x CANVAS_HEIGHT
    target_pil = target_pil.resize((FOLDED_WIDTH, CANVAS_HEIGHT), resample=Image.Resampling.LANCZOS)
    
    # 2. 转为 Tensor
    target_tensor = TF.to_tensor(target_pil).to(device).unsqueeze(0)
    
    print("\n目标图片处理完毕！这是折叠纸张后你会看到的画面：")
    rp.display_image(rp.as_numpy_image(target_tensor[0]))
else:
    print("❌ 未上传图片，请重新运行此块！")

In [ ]:
# === 🎮 游戏参数 ===
GUIDANCE_STRENGTH = 2500 # 隐写强度

# === 🎨 画面描述 ===
# Base: 完整的全景图 (展开时的样子)
prompt_base = "A wide panoramic landscape oil painting, beautiful mountains and rivers, continuous horizon, detailed, 4k"

negative_prompt = "blur, low quality, ugly, distortion, segmentation lines, seams, text, watermark"

# === 初始化可训练图像 ===
# 只需要一张图！
image_maker = lambda: LearnableImageFourier(height=CANVAS_HEIGHT, width=CANVAS_WIDTH, hidden_dim=256, num_features=256).to(device)
raw_image = image_maker()

# 定义获取图像的函数
get_image = lambda: raw_image()

# 准备标签
label_base = NegativeLabel(prompt_base, negative_prompt)

# 优化器
optim = torch.optim.SGD(raw_image.parameters(), lr=1e-4)

print("初始化完成。准备开始制作折纸幻觉！")

In [ ]:
NUM_ITER = 3000           
DISPLAY_INTERVAL = 200    

display_eta = rp.eta(NUM_ITER, title='Training Status')

print(f"🚀 开始训练... 目标：展开是一幅画，折叠后显现隐藏信息。")

history = []

try:
    for iter_num in range(NUM_ITER):
        display_eta(iter_num)

        # 1. 获取当前生成的完整大图
        curr_image = get_image()

        # --- Loss A: 展开状态要像一幅画 (SD Loss) ---
        # 让整张图符合 prompt_base (风景画)
        loss_sd = model_sd.train_step(
            label_base.embedding,
            curr_image[None],
            noise_coef=0.1,
            guidance_scale=60
        )

        # --- Loss B: 折叠状态要像目标图 (Geometric Loss) ---
        # 模拟物理折叠：切掉中间，拼接左右
        folded_view = simulate_fold(curr_image[None])
        
        # 计算与 Target 的 MSE Loss
        loss_fold = torch.mean((folded_view - target_tensor)**2) * GUIDANCE_STRENGTH
        
        # 反向传播 (两个 Loss 自动叠加)
        loss_fold.backward()

        # --- C. 显示进度 ---
        with torch.no_grad():
            if iter_num % DISPLAY_INTERVAL == 0:
                from IPython.display import clear_output
                clear_output(wait=True)
                
                # 转 Numpy 用于显示
                full_np = rp.as_numpy_image(curr_image)
                folded_np = rp.as_numpy_image(folded_view[0])
                
                # 给全图画上红线，方便看哪里会折叠
                full_with_lines = visualize_fold_lines(full_np)

                print(f"Iteration {iter_num} / {NUM_ITER}")
                print(f"图1: 完整展开图 (红线为折痕位置) | 图2: 折叠后的效果 (左+右拼接)")
                
                # 简单分行显示
                rp.display_image(full_with_lines)
                rp.display_image(folded_np)

        optim.step()
        optim.zero_grad()

except KeyboardInterrupt:
    print("用户手动停止训练。")

In [ ]:
print("==== 最终折纸成果 ====")
final_img = get_image()

# 1. 保存完整图
print("1. 打印这张图（沿着红线折叠，把中间部分盖住）：")
full_vis = visualize_fold_lines(rp.as_numpy_image(final_img))
rp.display_image(full_vis)

# 2. 展示折叠逻辑
print("\n2. 物理交互逻辑：")
print("   [ 左边 1/3 ]  <--折叠覆盖-->  [ 中间 1/3 ]  (对接)  [ 右边 1/3 ]")

# 3. 最终效果
print("\n3. 折叠后你将看到：")
folded_final = simulate_fold(final_img[None])[0]
rp.display_image(rp.as_numpy_image(folded_final))